# Het super sociaal netwerk

In deze notebook verken je verschillende technieken om inzicht te krijgen in een sociaal netwerk. Je zal bekijken hoe uitgespreid het netwerk is. Welke influencers er zijn in het netwerk en welke communities er zijn in het netwerk. 

Voor je start, installeer en importeer je de nodige bibliotheken.

In [ ]:
!pip install networkx

In [47]:
import csv
import numpy as np
import networkx as nx

## Het sociaal netwerk laden

In de map *dataset* kan je een bestand vinden met de naam *hero-network.csv*. Elke lijn van dit bestand bevat twee namen van superhelden. Zo'n paar werd aan het bestand toegevoegd wanneer de twee helden samen voorkomen in een strip. 

**Opdracht**: 
- Open het bestand *hero-network.csv* in de map *dataset*;
- Bekijk het bestand.
- Zijn er helden die je herkent? 
- Zijn er helden die je niet herkent?

Met de volgende codecel laden we de inhoud van het csv bestand in in een numpy array.

In [ ]:
# Lees het csv bestand.
data = None
with open('./dataset/hero-network-small-sample.csv', newline='') as csvfile:
    data = np.array(list(csv.reader(csvfile)))
    
# Verwijder de eerste rij, deze bevat de kolomnamen.
data = data[1:]

## Inzicht krijgen in de dataset

Voor je start met een analyse op een dataset, is het goed om een idee te krijgen van de gegevens die in die dataset zitten. Hieronder verkennen we een aantal eenvoudige eigenschappen van de dataset.

Voer de volgende codecel uit om 20 willekeurige heldenkoppels af te drukken.

In [ ]:
# Print 20 radom rijen van de data.
print(data[np.random.randint(0, len(data), 20)])

Door het formaat van onze numpy array af te drukken, komen we te weten hoeveel heldenkoppels er in de dataset zitten.

In [ ]:
# Druk het formaat af van de dataset.
print(f"Er zitten {data.shape[0]} duo's in de dataset.")

Wanneer je het csv bestand bekijkt, zal je zien dat er veel helden meerdere keren in de dataset voorkomen. Met de volgende codecel tellen we hoeveel unieke helden er in de dataset voorkomen.

In [ ]:
# Verzamel de unieke helden in een lijst.
helden = np.unique(data)
print(f"Er zitten {len(helden)} unieke helden in de dataset.")

We kunnen de helden ook alfabetisch afdrukken.

In [ ]:
helden_alfabetisch = list(np.sort(helden))
for held in helden_alfabetisch[:20]:
    print(f"{held}")

Om te weten in hoeveel strips twee helden met elkaar in contact komen, tellen we het aantal keer dat elk koppel voorkomt in de dataset. Merk op dat de volgorde waarin de namen van de helden in de dataset staan niet van belang is. Het koppel (Captain America, The Hulk) is hetzelfde als (The Hulk, Captain Amerika).

In [71]:
# Maak een dictionary die het aantal interacties per koppel helden bijhoudt.

# Initialiseer de dictionary.
interacties = {}

# Overloop alle heldenkoppels
for rij in data:
    # Sorteer de namen van het koppel zodat de volgorde niet uitmaakt.
    heldenpaar = tuple(sorted(rij))
    # Voeg het heldenpaar toe aan de dictionary of verhoog het aantal interacties.
    if heldenpaar in interacties:
        interacties[heldenpaar] += 1
    else:
        interacties[heldenpaar] = 1
        

Hieronder drukken we af hoe vaak de eerste 10 heldenparen voorkomen in de dataset.

In [ ]:
for paar in list(interacties.keys())[:20]:
    print(f"{paar[0]} en {paar[1]} interageren {interacties[paar]} keer.")

## Een graaf opstellen

Om een graaf te bouwen maken we gebruik van de networkx bibliotheek. Deze bibliotheek maakt het gemakkelijk om analyses te doen op grafen.

Eerst en vooral maken we een graafobject aan. Aan dit object gaan we onze knopen en bogen toevoegen.

In [73]:
# Maak een graafobject aan.
graaf = nx.Graph()

We kunnen op een eenvoudige manier onze helden toevoegen als knopen aan de graaf. Daarvoor gebruiken we de functie `add_nodes_from`. We geven een lijst mee aan deze functie. De functie zal dan voor elk element uit de lijst een knoop toevoegen.

In [74]:
# Voeg de helden toe als knopen.
graaf.add_nodes_from(helden)

Nu moeten we enkel nog de bogen toevoegen aan de graaf. Daarvoor kunnen we onze dictionary met interacties gebruiken. Elke interactie komt overeen met een boog tussen de knopen van de twee helden in het paar. Het gewicht van de boog is het aantal interacties die dat paar heeft.

In [75]:
# Voeg de interacties toe als gewogen bogen.
for paar in interacties:
    graaf.add_edge(paar[0], paar[1], weight=interacties[paar])

## Influencers opsporen

Zoals je eerder in het leerpad zag, kan je belangrijke mensen in een sociaal netwerk opsporen aan de hand van hun **graadcentraliteit**. Networkx heeft een eenvoudige methode om de graadcentraliteit van elke knoop te bepalen. Onderstaande code drukt de 10 helden af met hoogste graadcentraliteit.

In [ ]:
graadcentraliteit = nx.degree_centrality(graaf)
gesorteerde_graadcentraliteit = sorted(graadcentraliteit.items(), key=lambda x: x[1], reverse=True)

for held, graad in gesorteerde_graadcentraliteit[:10]:
    print(f"{held} heeft een graadcentraliteit van {graad}.")

**Opdracht**: Naast graadcentraliteit kan je ook de eigenwaardecentraliteit bepalen met de functie `eigenvector_centrality`. Gebruik onderstaande codecel om de 10 populairste helden af te drukken volgens eigenwaardecentraliteit.

**Opdracht**: Vergelijk de populairste helden volgens eigenwaarde- en graadcentraliteit. Welke verschillen zie je?

## Spreiding van het netwerk

De spreiding van het netwerk is de maximale kortste afstand tussen elke twee helden in het netwerk. Voor elk paar van knopen berekenen we met het algoritme van Dijkstra de lengte van het kortste pad tussen deze twee knopen. Van al deze afstanden nemen we de langste afstand als *diameter* van het netwerk. Deze diameter zegt ons iets over hoe uitgespreid het netwerk is. Hoe kleiner de diameter, hoe sterker de knopen in het netwerk met elkaar verbonden zijn.




**Merk op dat de code in de volgende cel veel tijd in beslag kan nemen. De computer moet immers de afstand berekenen tussen elk paar knopen in de graaf er zijn zo'n 2220 x 2220 = 4928400 mogelijke paren.**

In [ ]:
# Calculate the maximal shortest path between any two nodes
diameter = nx.diameter(graaf)
print(f"The diameter van de graaf is {diameter}")

## Communities zoeken

Naast de influencers in het netwerk en de spreiding van het netwerk, kunnen we ook op zoek gaan naar communities binnen het netwerk. In een sociaal netwerk komt een community vaak overeen met een specifieke vriendengroep. Bijvoorbeeld mensen die elkaar kennen van in de sportclub.

Om communities te detecteren maken we gebruik van de Louvain methode. Dit is een algoritme die stapsgewijs op zoek zal gaan naar communities binnen het netwerk. Dit doet het algoritme door de *modulariteit* van het netwerk te berekenen en deze stap voor stap te maximaliseren. Hier gebruiken we een speciale bibliotheek die al een implementatie van het algoritme bevat.

Eerst installeren we de bibliotheek.

In [ ]:
!pip install python-louvain

Dan importeren we de bibliotheek.

In [85]:
import community

Met deze bibliotheek kunnen we op een eenvoudige manier op zoek gaan naar "vriendengroepen" in ons netwerk.

In [86]:
vriendengroepen = community.best_partition(graaf)

Eerst en vooral bekijken we hoeveel vriendengroepen het algoritme gevonden heeft.

In [ ]:
# Druk het aantal vriendengroepen in het netwerk af.
print(f"Er zijn {len(set(vriendengroepen.values()))} vriendengroepen in het netwerk.")

We kunnen ook bekijken hoeveel helden in elke vriendengroep zitten.

In [ ]:
# Druk voor elke vriendengroep het aantal leden af.
for vriendengroep in set(vriendengroepen.values()):
    leden = [held for held, groep in vriendengroepen.items() if groep == vriendengroep]
    print(f"Vriendengroep {vriendengroep} heeft {len(leden)} leden.")

Om de populairste helden in elke vriendengroep te vinden, kunnen we de helden binnen een vriendengroep rangschikken volgens graadcentraliteit. Onderstaande cel drukt voor elke vriendengroep de 5 populairste helden af.

In [ ]:
# Druk de eerste 5 leden van elke vriendengroep af gerangschikt volgens graadcentraliteit.
for vriendengroep in set(vriendengroepen.values()):
    leden = [held for held, groep in vriendengroepen.items() if groep == vriendengroep]
    graadcentraliteit_van_leden = {held: graadcentraliteit[held] for held in leden}
    gesorteerde_leden = sorted(graadcentraliteit_van_leden.items(), key=lambda x: x[1], reverse=True)
    print(f"Vriendengroep {vriendengroep} heeft de volgende top 5 leden:")
    for held, graad in gesorteerde_leden[:5]:
        print(f"    - {held} met een graadcentraliteit van {graad}.")

**Opdracht:** Bekijk de uitvoer van bovenstaande cel. Hoe worden de vriendengroepen volgens jou gegroepeerd? Begrijp je waarom deze helden in dezelfde vriendengroep zitten?

# Partners

Dit materiaal werd ontwikkeld door Dwengo vzw en kwam tot stand met steun van VLAIO. Vind al het materiaal van ons wAIsda project op [dwengo.org/waisda](dwengo.org/waisda)

!["VLAIO logo"](img/vlaio.png)

!["Dwengo logo"](img/dwengo-groen-zwart.png)